# ADNI secondary-analysis mediation model

Tests: **GLP-1RA medication use → HOMA-IR (peripheral insulin resistance) → cognitive/MRI decline**, using real ADNI tabular data.

**Read this before running anything:**
- This is an *observational* secondary-data analysis, not a clinical trial. GLP-1RA "exposure" means a patient happened to already be on the drug (almost always for diabetes) — not random assignment. Any result here is an association consistent with the mediation hypothesis, not proof of it.
- HOMA-IR is a **peripheral** (blood) insulin-resistance proxy, not a measure of **central** (brain) insulin resistance, which is what the original research question asks about.
- ADNI has no Parkinson's disease arm — this notebook can only speak to Alzheimer's/MCI.
- You need your own approved ADNI data access. Apply at https://adni.loni.usc.edu (free, ~1–2 day approval), then download from the LONI Image and Data Archive (IDA):
  1. `ADNIMERGE.csv` — demographics, diagnosis, cognition (ADAS13), MRI volumes (Hippocampus, ICV)
  2. `RECCMEDS.csv` — recent/concomitant medication log
  3. A lab/biomarker file containing fasting **glucose AND insulin** — search the IDA Data Dictionary for "insulin"; the exact filename varies by ADNI phase/ancillary study, so it can't be hardcoded.

**Until you have that access**, run this notebook in `DEMO` mode — it uses a synthetic, schema-matched dataset (clearly fake) so you can see exactly how the pipeline and plots work end to end.

## 1. Setup

In [ ]:
# Clone your repo (replace with your actual GitHub URL) OR upload the analysis/ folder manually via the Colab file browser on the left.
# !git clone https://github.com/<your-username>/<your-repo>.git
# %cd <your-repo>/glp1-insulin-resistance/analysis

!pip install -q pandas numpy scipy statsmodels matplotlib seaborn

In [ ]:
# If your ADNI CSVs are in Google Drive, mount it here. Skip this cell in DEMO mode.
from google.colab import drive
drive.mount('/content/drive')

## 2. Choose mode: DEMO (synthetic, works right now) or REAL (your ADNI files)

In [ ]:
MODE = "DEMO"  # change to "REAL" once you have ADNI files uploaded/mounted

# Only used when MODE == "REAL" — update these paths to wherever you placed your downloaded ADNI files
ADNIMERGE_PATH = "/content/drive/MyDrive/ADNI/ADNIMERGE.csv"
MEDICATIONS_PATH = "/content/drive/MyDrive/ADNI/RECCMEDS.csv"
BIOMARKERS_PATH = "/content/drive/MyDrive/ADNI/insulin_glucose.csv"
GLUCOSE_COL = "GLUCOSE"   # adjust to match your actual biomarker file's column name
INSULIN_COL = "INSULIN"   # adjust to match your actual biomarker file's column name
OUTCOME = "adas13"        # or "hippocampus"

In [ ]:
import sys
sys.path.insert(0, ".")  # so adni_pipeline.py / glp1_model.py are importable; adjust if your files live elsewhere

import adni_pipeline as ap
import glp1_model as mm  # mediation_model.py was merged into glp1_model.py; same API
import pandas as pd
from pathlib import Path

outdir = Path("output/adni_colab")
outdir.mkdir(parents=True, exist_ok=True)

## 3. Load data and build the cohort

In [ ]:
if MODE == "DEMO":
    print("=== DEMO MODE: synthetic schema-matched data, NOT real ADNI data ===")
    adnimerge, medications, biomarkers = ap.generate_synthetic_adni_demo(seed=42)
    glp1ra_flags = ap.flag_glp1ra_users(medications)
    homa_ir = biomarkers.assign(homa_ir=ap.compute_homa_ir(biomarkers["GLUCOSE"], biomarkers["INSULIN"]))[["RID", "homa_ir"]]
    cohort = ap.build_cohort(adnimerge, glp1ra_flags, homa_ir, outcome="adas13")
else:
    adnimerge = ap.load_adnimerge(ADNIMERGE_PATH)
    medications = pd.read_csv(MEDICATIONS_PATH, low_memory=False)
    glp1ra_flags = ap.flag_glp1ra_users(medications)
    homa_ir = ap.load_biomarkers(BIOMARKERS_PATH, glucose_col=GLUCOSE_COL, insulin_col=INSULIN_COL)
    cohort = ap.build_cohort(adnimerge, glp1ra_flags, homa_ir, outcome=OUTCOME)

n_treated = int(cohort["treatment"].sum())
print(f"Cohort assembled: n={len(cohort)} total, {n_treated} GLP-1RA users, {len(cohort) - n_treated} non-users")
if n_treated < 10:
    print("WARNING: fewer than 10 GLP-1RA users — mediation estimates will be very unstable. "
          "This is the expected limitation of incidental GLP-1RA exposure in an observational cohort.")
cohort.head()

## 4. Run the mediation model

In [ ]:
covariates = [c for c in ["age", "sex_male", "education", "apoe4_carrier"] if cohort[c].notna().all()]

result = mm.fit_mediation(
    cohort, treatment="treatment", mediator="mediator", outcome="outcome_change",
    covariates=covariates, n_boot=5000, seed=42,
    notes=["Observational/incidental GLP-1RA exposure, not randomized.",
           "HOMA-IR is a peripheral proxy, not a central/brain insulin-resistance measure."],
)
print(result.summary())

## 5. Visualize results

In [ ]:
outcome_label = "Change in ADAS13 (higher = worse)" if OUTCOME == "adas13" else "Change in Hippocampus/ICV (higher = better)"

mm.plot_path_diagram(result, "GLP-1RA use", "HOMA-IR", outcome_label,
                      outdir / "adni_path_diagram.png", title="ADNI secondary analysis: mediation path model")
ap.plot_cohort_overview(cohort, outcome_label, outdir / "adni_cohort_overview.png")
ap.plot_bootstrap_distribution(result, outdir / "adni_bootstrap_distribution.png")

In [ ]:
from IPython.display import Image, display
display(Image(str(outdir / "adni_path_diagram.png")))
display(Image(str(outdir / "adni_cohort_overview.png")))
display(Image(str(outdir / "adni_bootstrap_distribution.png")))

## 6. How to read the output

- **Path a** (GLP-1RA → HOMA-IR): does GLP-1RA use associate with lower HOMA-IR in this cohort?
- **Path b** (HOMA-IR → outcome, controlling for treatment): does HOMA-IR associate with worse cognitive/MRI outcomes, independent of treatment?
- **Indirect effect (a×b)** and its 95% bootstrap CI: the part of the GLP-1RA → outcome relationship that runs *through* HOMA-IR. If the CI excludes zero, that's *consistent with* mediation — not proof, given the observational design and the peripheral-vs-central IR gap noted above.
- **Proportion mediated**: how much of the total treatment effect the HOMA-IR pathway could account for.

Whatever you find here, report it as a hypothesis-generating secondary-data finding, alongside its limitations (observational confounding by diabetes severity/duration, small/incidental GLP-1RA-user subgroup, peripheral-not-central IR marker, AD-only cohort).